# 08 | Executive Insights & Decision Brief

**Project:** Dubai Short-Term Rental Intelligence · **Source:** AirROI Dubai third-party dataset · **Reporting window:** March 2021–February 2026

**Primary stakeholders:** Short-term rental property managers, revenue analysts, and business intelligence teams.

**Decision question:** Which properties should management investigate for potential pricing misalignment, how reliable are their peer benchmarks, and what evidence is required before taking action?

This final analytical notebook consolidates *verified results* from the relational dataset built and validated in Notebooks 03–07. It produces a stakeholder-oriented decision brief and two concise visualizations. **This is a retrospective sample analysis, not a live view of the Dubai rental market.** Reported occupancy and revenue are provider estimates, not verified reservations, realized revenue, or profit.

## 1. Reproducible evidence and reporting controls

**Authoritative source:** `data/processed/dubai_rentals.sqlite` generated by Notebook 05 and validated by Notebook 07. The notebook works from the repository root or `notebooks/`. Historical JSON summaries from earlier stages are **optional cross-checks**: missing files are reported as not checked, never silently fabricated. If a present summary disagrees with the database, validation fails.

**Dependencies:** `pandas`, `numpy`, `matplotlib`, `IPython`; Python's built-in `sqlite3` and `json` modules. No private data or absolute machine-specific paths are embedded.

**Deliverables:** `reports/business_insights_summary.md`, `reports/notebook08_validation.csv`, `reports/notebook08_evidence.json`, and two charts in `reports/figures/`. All derived counts in the report are computed at runtime.

**Snapshot control (independent audit, 10 October 2026):** Fixed figures in this narrative refer to the four original files identified by SHA-256 in `reports/source_audit.json`. Executable tables derive values at runtime. Review the narrative if replacing the source snapshot. Raw data and listing-level outputs remain private; conclusions describe this sample and provider estimates.

In [3]:
from pathlib import Path
import sqlite3
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, cwd.parent, cwd.parent.parent)
                     if (p / 'README.md').is_file() and (p / 'data' / 'processed').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open Jupyter from the repository root or notebooks/ directory.')
DB = PROJECT_ROOT / 'data' / 'processed' / 'dubai_rentals.sqlite'
REPORTS = PROJECT_ROOT / 'reports'
FIGURES = REPORTS / 'figures'
if not DB.is_file():
    raise FileNotFoundError(f'Missing database: {DB.relative_to(PROJECT_ROOT)}. Run Notebooks 03–05 before Notebook 08.')
FIGURES.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 24)
print('Source:', DB.relative_to(PROJECT_ROOT), '| size (MiB):', round(DB.stat().st_size / 1024**2, 1))

FileNotFoundError: Open Jupyter from the repository root or notebooks/ directory.

## 2. Reconcile the historical dataset and latest reporting period

Metrics are calculated directly from `fact_listing_month` and `fact_peer_benchmark`, with the latest month resolved dynamically. A snapshot count is not a market census; comparison of historical row totals is not a measure of market growth when sample coverage changes.

In [ ]:
with sqlite3.connect(DB) as conn:
    meta = pd.read_sql_query("""
        SELECT COUNT(*) AS history_rows, COUNT(DISTINCT listing_id) AS historical_listings,
               COUNT(DISTINCT month_date) AS months,
               MIN(month_date) AS first_month, MAX(month_date) AS latest_month
        FROM fact_listing_month
    """, conn).iloc[0]
    latest_month = str(meta['latest_month'])
    latest_rows = pd.read_sql_query("""
        SELECT f.listing_id, f.active, f.neighborhood, f.native_avg_daily_rate,
               f.occupancy, f.native_revenue, f.room_type, f.bedrooms,
               b.peer_count, b.radius_km, b.premium_pct, b.peer_median_aed,
               b.comparison_status, b.review_signal, b.confidence_label
        FROM fact_listing_month AS f
        JOIN fact_peer_benchmark AS b
          ON b.listing_id = f.listing_id AND b.month_date = f.month_date
        WHERE f.month_date = ?
    """, conn, params=(latest_month,))
    coverage = pd.read_sql_query("""
        SELECT month_date, observed_listings, active_listings,
               coverage_change_pct, same_listing_prev_month
        FROM fact_coverage_month ORDER BY month_date
    """, conn)

active_rows = latest_rows[latest_rows['active'].eq(1)].copy()
benchmarked = latest_rows[latest_rows['comparison_status'].eq('Eligible')].copy()
flagged = benchmarked[benchmarked['review_signal'].isin(('Review','Large deviation'))].copy()
above = flagged['premium_pct'].gt(0).sum()
below = flagged['premium_pct'].lt(0).sum()
counts = {
    'historical_rows': int(meta['history_rows']),
    'unique_properties': int(meta['historical_listings']),
    'months': int(meta['months']),
    'first_month': str(meta['first_month']), 'latest_month': latest_month,
    'observed': len(latest_rows), 'active': len(active_rows),
    'median_active_rate': float(active_rows['native_avg_daily_rate'].median()),
    'zero_occupancy_active': int(active_rows['occupancy'].eq(0).sum()),
    'missing_neighborhood': int(latest_rows['neighborhood'].fillna('').astype(str).str.strip().eq('').sum()),
    'benchmarked': len(benchmarked),
    'peer_eligible': int(latest_rows['comparison_status'].isin(('Eligible','Insufficient peers')).sum()),
    'insufficient_peers': int(latest_rows['comparison_status'].eq('Insufficient peers').sum()),
    'not_peer_eligible': int(latest_rows['comparison_status'].eq('Ineligible').sum()),
    'flagged': len(flagged), 'above': int(above), 'below': int(below),
    'unassessed': int(latest_rows['review_signal'].eq('Unassessed').sum()),
    'within_threshold': int(benchmarked['review_signal'].eq('Within threshold').sum()),
    'large_deviation': int(benchmarked['review_signal'].eq('Large deviation').sum()),
    'moderate_deviation': int(benchmarked['review_signal'].eq('Review').sum()),
    'coverage_shifts_25': int(coverage['coverage_change_pct'].abs().gt(25).sum()),
}
counts['zero_occupancy_pct'] = 100 * counts['zero_occupancy_active'] / counts['active']
counts['missing_neighborhood_pct'] = 100 * counts['missing_neighborhood'] / counts['observed']
counts['flagged_of_benchmarked_pct'] = 100 * counts['flagged'] / counts['benchmarked']
counts['benchmarked_of_active_pct'] = 100 * counts['benchmarked'] / counts['active']

snapshot = pd.DataFrame([
    ('Historical listing-month observations', f"{counts['historical_rows']:,}"),
    ('Historical unique properties', f"{counts['unique_properties']:,}"),
    ('Monthly periods', counts['months']),
    ('Latest reporting month', counts['latest_month']),
    ('Latest observed listings', f"{counts['observed']:,}"),
    ('Latest active listings', f"{counts['active']:,}"),
    ('Active median reported nightly rate', f"AED {counts['median_active_rate']:,.0f}"),
    ('Active listings with zero provider-estimated occupancy',
     f"{counts['zero_occupancy_active']:,} ({counts['zero_occupancy_pct']:.1f}%)"),
    ('Observed listings with missing neighborhood',
     f"{counts['missing_neighborhood']:,} ({counts['missing_neighborhood_pct']:.1f}%)"),
], columns=['Indicator','Result'])
display(snapshot)

## 3. Benchmark coverage and pricing review classifications

Peers are matched within the same reporting month and accommodation type/bedroom count. The algorithm first searches within **2 km**, then expands to **5 km** if fewer than **10 peers** are available. The benchmark is the *median reported nightly rate* among selected peers. Price deviations of **20% or more** trigger a review flag; deviations of **35% or more** trigger a large-deviation flag. These are *analyst-defined screening rules*, not evidence that a price is wrong or that a change would increase revenue.

The **benchmark denominator** and **review denominator** are shown explicitly so the project does not confuse ineligible/unassessed listings with those whose rates fall within the thresholds.

In [ ]:
review_rows = pd.DataFrame([
    ('Above peer median (review required)', counts['above']),
    ('Below peer median (review required)', counts['below']),
    ('Within review threshold', counts['within_threshold']),
    ('Unassessed: ineligible or insufficient peers', counts['unassessed']),
], columns=['Pricing classification', 'Listings'])
coverage_rows = pd.DataFrame([
    ('Valid benchmarks (active)', counts['benchmarked']),
    ('Active without valid benchmark', counts['active'] - counts['benchmarked']),
    ('Inactive in latest snapshot', counts['observed'] - counts['active']),
], columns=['Benchmark availability', 'Listings'])
display(review_rows)
display(coverage_rows)
print(f"Review flags: {counts['flagged']:,} of {counts['benchmarked']:,} benchmarked listings "
      f"({counts['flagged_of_benchmarked_pct']:.1f}%).")
print(f"Valid benchmarks: {counts['benchmarked']:,} of {counts['active']:,} active listings "
      f"({counts['benchmarked_of_active_pct']:.1f}%).")

## 4. Independent validation and summary-version checks

These checks reconcile key totals from the database, verify the decision thresholds, and detect stale or contradictory upstream summaries. **Optional** JSON files are cross-checked if present; their absence is marked `SKIP`, not treated as fabricated evidence. A `FAIL` blocks report generation.

In [ ]:
checks = []
def check(name, condition, detail=''):
    checks.append({'check': name, 'status': 'PASS' if bool(condition) else 'FAIL', 'detail': detail})

def approximately_equal(a, b, tol=1e-6):
    return pd.notna(a) and pd.notna(b) and abs(float(a) - float(b)) <= tol

with sqlite3.connect(DB) as conn:
    db_integrity = conn.execute('PRAGMA integrity_check').fetchone()[0]
    foreign_key_issues = conn.execute('PRAGMA foreign_key_check').fetchall()
    sql_latest_count = conn.execute('SELECT COUNT(*) FROM fact_listing_month WHERE month_date=?',
                                    (latest_month,)).fetchone()[0]
    sql_latest_flagged = conn.execute('SELECT COUNT(*) FROM vw_latest_price_review_queue').fetchone()[0]
    sql_benchmarks = conn.execute('SELECT COUNT(*) FROM fact_peer_benchmark').fetchone()[0]
    sql_dupes = conn.execute("""SELECT COUNT(*) FROM (
        SELECT listing_id,month_date FROM fact_listing_month
        GROUP BY listing_id,month_date HAVING COUNT(*)>1)""").fetchone()[0]
    sql_active = conn.execute('SELECT COUNT(*) FROM fact_listing_month WHERE month_date=? AND active=1',
                              (latest_month,)).fetchone()[0]
    sql_coverage = pd.read_sql_query("""
        SELECT month_date, COUNT(*) AS observed_listings, SUM(active) AS active_listings
        FROM fact_listing_month GROUP BY month_date ORDER BY month_date
    """, conn)

check('SQLite integrity', db_integrity == 'ok', db_integrity)
check('SQLite foreign keys', not foreign_key_issues, f'{len(foreign_key_issues)} violations')
check('No duplicate listing-month keys', sql_dupes == 0, f'{sql_dupes} duplicates')
check('Fact and benchmark history reconciled', sql_benchmarks == counts['historical_rows'])
check('Latest month rows reconcile', sql_latest_count == counts['observed'])
check('Active rows reconcile', sql_active == counts['active'])
check('Observed >= active >= benchmarked', counts['observed'] >= counts['active'] >= counts['benchmarked'])
check('Benchmark status partitions latest snapshot',
      counts['benchmarked'] + counts['insufficient_peers'] + counts['not_peer_eligible'] == counts['observed'])
check('Pricing classifications partition snapshot', int(review_rows['Listings'].sum()) == counts['observed'])
check('Benchmark review queue reconciles', sql_latest_flagged == counts['flagged'])
check('Flagged listing directions reconcile', counts['above'] + counts['below'] == counts['flagged'])
check('Review flags follow >=20% screening threshold',
      flagged['premium_pct'].notna().all() and flagged['premium_pct'].abs().ge(20-1e-8).all())
check('Large deviation flags follow >=35% threshold',
      benchmarked.loc[benchmarked['review_signal'].eq('Large deviation'),'premium_pct'].abs().ge(35-1e-8).all())
check('Within threshold flags are below 20%',
      benchmarked.loc[benchmarked['review_signal'].eq('Within threshold'),'premium_pct'].abs().lt(20+1e-8).all())
check('Valid peer groups have at least ten peers',
      benchmarked['peer_count'].ge(10).all() and benchmarked['peer_median_aed'].gt(0).all())
check('Latest month matches coverage table', coverage['month_date'].iloc[-1] == latest_month)
check('Monthly coverage observation reconciliation',
      bool(len(sql_coverage) == len(coverage) and sql_coverage.merge(coverage, on='month_date', suffixes=('_fact','_coverage'))
           .eval('observed_listings_fact == observed_listings_coverage and active_listings_fact == active_listings_coverage').all()))
check('History months have unique coverage records',
      coverage['month_date'].is_unique and len(coverage) == counts['months'])

summaries = [('data_quality_summary.json', {
    'primary_rows': counts['historical_rows'], 'latest_active': counts['active'],
    'latest_observed': counts['observed'], 'latest_month': latest_month,
    'month_count': counts['months']}),
    ('benchmark_latest_summary.json', {
    'month': latest_month, 'active_sample': counts['active'],
    'eligible_for_matching': counts['peer_eligible'],
    'with_valid_benchmark': counts['benchmarked']})]
for filename, expected in summaries:
    path = REPORTS / filename
    if not path.exists():
        checks.append({'check': f'Optional summary {filename}', 'status': 'SKIP',
                       'detail': 'Not present; current database is the authoritative source'})
        continue
    original = json.loads(path.read_text(encoding='utf-8'))
    for key, value in expected.items():
        found = original.get(key)
        same = (approximately_equal(found, value) if isinstance(value, (int,float)) and not isinstance(value,bool)
                else found == value)
        check(f'{filename}: {key}', same, f'DB={value}, JSON={found}')

validation = pd.DataFrame(checks)
display(validation)
print('Results:', validation['status'].value_counts().to_dict())
if validation['status'].eq('FAIL').any():
    raise AssertionError('Data validation failed; resolve discrepancies before generating the executive brief.')
validation.to_csv(REPORTS / 'notebook08_validation.csv', index=False)

## 5. Management visualizations

Charts show the latest dataset snapshot and use observed listing counts, not hypothetical values or weighted estimates. The two visuals distinguish *pricing review status* from *eligibility and availability of benchmarks*.

In [ ]:
# Chart 1 — mutually exclusive review classifications for all observed latest-month listings
labels = ['Above peers: review', 'Below peers: review', 'Within threshold', 'Unassessed']
values = [counts['above'], counts['below'], counts['within_threshold'], counts['unassessed']]
colors = ['#D97735', '#2C8C98', '#65788C', '#B4BEC7']
fig, ax = plt.subplots(figsize=(10,4.6))
ax.barh(labels[::-1], values[::-1], color=colors[::-1], height=.66)
for i,v in enumerate(values[::-1]):
    ax.text(v+max(values)*.018, i, f'{v:,}', va='center', fontsize=11, weight='bold', color='#203247')
ax.set_xlim(0, max(values)*1.17)
ax.set_xlabel('Number of observed listings')
ax.set_title(f'Pricing review classifications | {latest_month[:7]}', weight='bold', pad=15)
ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(axis='y',length=0)
ax.grid(axis='x',alpha=.13)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / '08_pricing_review_classification.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# Chart 2 — how the latest observed properties contribute to usable benchmarks
coverage_labels = ['Valid peer benchmark', 'Active without benchmark', 'Inactive']
coverage_values = [counts['benchmarked'], counts['active']-counts['benchmarked'], counts['observed']-counts['active']]
coverage_colors = ['#198F8A', '#E0A357', '#B4BEC7']
fig, ax = plt.subplots(figsize=(10,3.4))
left = 0
for value, label, color in zip(coverage_values, coverage_labels, coverage_colors):
    ax.barh(['Latest snapshot'], [value], left=left, color=color, height=.45, label=f'{label}: {value:,}')
    if value > counts['observed'] * .09:
        ax.text(left + value/2, 0, f'{value:,}', ha='center', va='center',
                color='white' if color != '#B4BEC7' else '#203247', weight='bold', fontsize=11)
    left += value
ax.set_xlim(0, counts['observed'])
ax.set_xlabel('Observed listings (not Dubai market total)')
ax.set_title(f'Benchmark coverage | {latest_month[:7]}', weight='bold', pad=15)
ax.legend(loc='upper center', bbox_to_anchor=(.5,-.35),ncol=3,frameon=False,fontsize=9)
ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(axis='y',length=0)
ax.grid(axis='x',alpha=.12)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / '08_benchmark_coverage.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Stakeholder decisions and evidence requirements

These are **recommended investigations**, not demonstrated operational improvements. Changing an advertised rate may not improve occupancy or realized revenue, and the snapshot does not establish causality.

| Stakeholder | Priority question | Evidence-based next action | What must be verified first |
|:--|:--|:--|:--|
| Revenue management | Why is a listing priced ≥20% above its peer benchmark? | Inspect the comparable set and explain the gap before considering changes | Amenities, quality, listing restrictions, event dates, actual booking outcomes |
| Property management | Why is a listing priced ≥20% below peers? | Review positioning and assess whether the gap reflects quality or strategy | Listing amenities, customer ratings, channel fees, confirmed reservations |
| Operations management | Why is provider-estimated occupancy zero? | Flag for external verification, not a vacancy conclusion | Actual reservations, blocked dates, seasonality, property status |
| BI / data governance | Which listings lack valid peers? | Improve critical source attributes and show benchmark coverage in report | Location, bedrooms, room type, observations per peer set |
| Market analysis | Are observed historical changes compositional? | Use matched listing cohorts and prominently display monthly sample counts | Stable coverage and consistent measurement definitions |

In [ ]:
# Produce a succinct analytical prioritization without claiming causal value.
summary_groups = (flagged.assign(direction=np.where(flagged['premium_pct']>=20, 'Above peers','Below peers'))
                  .groupby(['review_signal','direction'],dropna=False)
                  .agg(listings=('premium_pct','size'),
                       median_absolute_deviation_pct=('premium_pct',lambda s: s.abs().median()),
                       median_peer_count=('peer_count','median'))
                  .reset_index().sort_values(['review_signal','direction']))
display(summary_groups)

sample_shifts = (coverage.loc[coverage['coverage_change_pct'].abs().gt(25),
                              ['month_date','observed_listings','coverage_change_pct']]
                         .sort_values('month_date'))
display(sample_shifts.style.format({'coverage_change_pct': '{:+.1f}%'}))

## 7. Publish the executive decision brief

The Markdown report is generated from the SQL-validated evidence in this notebook. It includes key denominators, a segmented review summary, operational decisions, and explicit uncertainty. Charts are saved as local project assets for use on GitHub. No raw row-level source data is included.

In [ ]:
from datetime import datetime, timezone

def n(v): return f'{int(v):,}'
def pct(a, total): return 100 * a / total if total else float('nan')

segment_lines = ['| Review signal | Direction | Listings | Median absolute deviation | Median peer count |',
                 '|---|---|---:|---:|---:|']
for r in summary_groups.itertuples(index=False):
    segment_lines.append(f'| {r.review_signal} | {r.direction} | {n(r.listings)} | '
                         f'{r.median_absolute_deviation_pct:.1f}% | {r.median_peer_count:.0f} |')

lines = [
 '# Dubai Short-Term Rental Intelligence — Executive Decision Brief', '',
 f'**Source:** AirROI third-party Dubai sample  |  **Latest month:** {latest_month[:7]}  |  '
 '**Purpose:** manual pricing-review prioritization', '',
 '## Executive summary', '',
 f'The study covers **{n(counts["historical_rows"])} property-month observations** from '
 f'**{counts["first_month"][:7]} to {latest_month[:7]}**, representing '
 f'**{n(counts["unique_properties"])} historical properties**. In the latest snapshot, '
 f'**{n(counts["observed"])}** listings were observed and **{n(counts["active"])}** were active. '
 f'Only **{n(counts["benchmarked"])} active listings ({counts["benchmarked_of_active_pct"]:.1f}%)** '
 f'had a valid peer benchmark under the model rules. Of these, **{n(counts["flagged"])} '
 f'({counts["flagged_of_benchmarked_pct"]:.1f}%)** were flagged for **manual pricing review**; '
 f'**{n(counts["above"])}** were above peer medians and **{n(counts["below"])}** below. '
 'These findings indicate investigation priorities, **not validated price optimization opportunities**.', '',
 '## Sample performance and data coverage', '',
 '| Measure | Latest snapshot | Interpretation |', '|---|---:|---|',
 f'| Observed listings | {n(counts["observed"])} | Snapshot coverage, not market size |',
 f'| Active listings | {n(counts["active"])} | Observed active listings |',
 f'| Median reported nightly rate | AED {counts["median_active_rate"]:,.0f} | Listed/provider-reported rate; not necessarily realized ADR |',
 f'| Active listings with zero estimated occupancy | {n(counts["zero_occupancy_active"])} ({counts["zero_occupancy_pct"]:.1f}%) | Provider-estimated, not verified empty nights |',
 f'| Missing neighborhood among observed listings | {n(counts["missing_neighborhood"])} ({counts["missing_neighborhood_pct"]:.1f}%) | Restricts area-based comparisons |',
 f'| Months with >25% absolute sample-count change | {n(counts["coverage_shifts_25"])} | Unadjusted historical changes risk coverage bias |', '',
 '## Competitor benchmark methodology', '',
 '- Same reporting month, room type and bedroom count; location determined using coordinates.',
 '- Find comparable listings within 2 km, expand to 5 km when fewer than 10 peers are available.',
 '- Only groups with at least 10 qualifying peers are benchmarked; subject listing is excluded.',
 '- Calculate median peer reported nightly rate and percentage difference from that median.',
 '- Absolute difference ≥20% triggers **Review**; ≥35% triggers **Large deviation**.',
 '- Rules are transparent heuristics and do not control completely for amenities, quality, policies, or realized demand.', '',
 '## Competitive pricing intelligence', '',
 f'- **{n(counts["peer_eligible"])}** listings had sufficient attributes to attempt matching; '
 f'**{n(counts["insufficient_peers"])}** of them lacked a large-enough peer set.',
 f'- **{n(counts["benchmarked"])}** received valid benchmarks; **{n(counts["within_threshold"])}** fell within the review threshold.',
 f'- **{n(counts["flagged"])}** benchmarked listings triggered a review flag '
 f'({n(counts["large_deviation"])} large deviations and {n(counts["moderate_deviation"])} moderate deviations).',
 f'- **{n(counts["unassessed"])}** of all latest observed listings were unassessed; do not interpret these as competitively priced.', '',
 '![Pricing review classifications](figures/08_pricing_review_classification.png)', '',
 '![Benchmark availability](figures/08_benchmark_coverage.png)', '',
 '### Segmentation of flagged properties', '',
 *segment_lines, '',
 '## Stakeholder recommendations', '',
 '| Priority | Owner | Recommended investigation | Additional evidence required |',
 '|---|---|---|---|',
 '| **1** | Revenue manager | Examine high deviations in comparable-listing context before any rate adjustment | Amenities, size, ratings, event dates, real bookings |',
 '| **2** | Property manager | Review below-peer properties for intentional positioning vs. listing disadvantages | Guest reviews, operating constraints, property condition |',
 '| **3** | Operations manager | Validate zero modeled occupancy and unusual performance externally | Booking platform exports and blocked-date rules |',
 '| **4** | BI / market analyst | Surface benchmark coverage and use stable listing cohorts for time comparisons | Attribute completeness and monthly sample retention |', '',
 '## Decision boundaries and limitations', '',
 '- Third-party AirROI data is **not an official Airbnb transaction dataset** or complete Dubai market census.',
 '- Estimated occupancy/revenue may not equal bookings, realized revenue or cash flow; occupancy = 0 is not verified vacancy.',
 '- A high or low price premium alone cannot establish whether the listing is overpriced or underpriced.',
 '- Peer matching is limited to month, proximity, room type and bedrooms; quality and amenities can differ.',
 '- Property composition varies across reporting periods; historical sample counts should accompany trend measures.',
 '- Neighborhood labels are often missing; coordinate proximity is not the same as a verified neighborhood boundary.',
 f'- Observations end in **{latest_month[:7]}**; findings do not describe live market conditions.',
 '- Underlying raw provider files are excluded from public distribution pending a redistribution-rights review.', '',
 '## Power BI implementation', '',
 'The **Executive Command Center** will show sample coverage and reported rates. The **Competitive Pricing Studio** '
 'will expose individual peer counts, matching radius and rate deviations. The **Opportunity & Action Center** '
 'will display review priorities alongside benchmark eligibility and data-quality caveats. The dashboard is a separate '
 'deliverable; completing this notebook does not establish that a Power BI report is built or validated.', '',
 '## Reproducibility', '',
 '`notebooks/08_insights_and_decision_brief.ipynb` derives these metrics from '
 '`data/processed/dubai_rentals.sqlite`. Optional summary JSON files are cross-checked when present; '
 'all internal validation checks must pass. See `reports/notebook08_validation.csv` and '
 '`reports/notebook08_evidence.json` for the test and evidence records.', ''
]
out = REPORTS / 'business_insights_summary.md'
out.write_text('\n'.join(lines), encoding='utf-8')
evidence = {
    'data_source':'AirROI Dubai third-party sample',
    'reporting_period': {'first':counts['first_month'], 'latest':counts['latest_month']},
    'measures':counts,
    'review_segments':summary_groups.to_dict(orient='records'),
    'validation':validation['status'].value_counts().to_dict(),
    'source_database':str(DB.relative_to(PROJECT_ROOT)),
    'generated_utc':datetime.now(timezone.utc).isoformat(),
}
(REPORTS / 'notebook08_evidence.json').write_text(json.dumps(evidence, indent=2),encoding='utf-8')
print('Created:',out.relative_to(PROJECT_ROOT))
print('Created:', (REPORTS/'notebook08_evidence.json').relative_to(PROJECT_ROOT))
print('Created:', (REPORTS/'notebook08_validation.csv').relative_to(PROJECT_ROOT))

## 8. Final analytical assessment

The latest snapshot includes a subset of observed properties with valid peer benchmarks, some of which trigger analyst-defined pricing review thresholds (see verified results above). This creates a reproducible *review workflow*, not evidence of an optimal nightly rate. Unassessed listings remain visible rather than being silently classified as within range. Sample volatility and provider-estimated performance prevent strong causal or market-wide conclusions.

**Stakeholder handoff:** Business teams should review property comparability and actual booking evidence before taking action. The Python and SQL analysis provides the foundation for the final Power BI report, where users can explore individual listings, compare prices with similar nearby properties, and review historical performance and data limitations.